In [ ]:
import pickle
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Miceimmune_parts.pkl", "rb") as f:
    payload = pickle.load(f)

adata_immune = ad.AnnData(
    X=payload["X"],
    obs=payload["obs"],
    var=payload["var"],
)

adata_immune.uns = payload["uns"]
adata_immune.obsm = payload["obsm"]
adata_immune.varm = payload["varm"]
adata_immune.layers = payload["layers"]

if "raw_X" in payload:
    adata_immune.raw = ad.AnnData(
        X=payload["raw_X"],
        var=payload["raw_var"]
    )

In [ ]:
with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Micefap_parts.pkl", "rb") as f:
    payload = pickle.load(f)

adata_fap = ad.AnnData(
    X=payload["X"],
    obs=payload["obs"],
    var=payload["var"],
)

adata_fap.uns = payload["uns"]
adata_fap.obsm = payload["obsm"]
adata_fap.varm = payload["varm"]
adata_fap.layers = payload["layers"]

if "raw_X" in payload:
    adata_fap.raw = ad.AnnData(
        X=payload["raw_X"],
        var=payload["raw_var"]
    )

In [ ]:
adata_fap.obs

In [ ]:
adata_immune = adata_immune.copy()

In [ ]:
adata_fap = adata_fap.copy()

In [ ]:
adata_immune.obs

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import liana as li

# -----------------------------
# 1. Keep only shared genes
# -----------------------------
shared_genes = adata_fap.var_names.intersection(adata_immune.var_names)

adata_fap_sub = adata_fap[:, shared_genes].copy()
adata_immune_sub = adata_immune[:, shared_genes].copy()

# -----------------------------
# 2. Make sure celltype + condition columns exist
# -----------------------------
for adata_obj in [adata_fap_sub, adata_immune_sub]:
    if "celltype" not in adata_obj.obs.columns:
        raise ValueError("Need a 'celltype' column in .obs")
    if "condition" not in adata_obj.obs.columns:
        raise ValueError("Need a 'condition' column in .obs")

# Optional: add source label
adata_fap_sub.obs["compartment"] = "FAP"
adata_immune_sub.obs["compartment"] = "Immune"

# -----------------------------
# 3. Concatenate into one object
# -----------------------------
adata_combined = ad.concat(
    [adata_fap_sub, adata_immune_sub],
    join="inner",
    label="dataset",
    keys=["fap", "immune"],
    merge="same"
)

# Make sure celltype is categorical
adata_combined.obs["celltype"] = adata_combined.obs["celltype"].astype("category")
adata_combined.obs["condition"] = adata_combined.obs["condition"].astype("category")

print(adata_combined)
print(adata_combined.obs["celltype"].value_counts())

In [ ]:
import liana as li

li.mt.cellphonedb(
    adata_combined,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    use_raw=False,
    n_perms=1000
)

In [ ]:
res = adata_combined.uns["liana_res"]
res.head()

In [ ]:
res[
    (res["ligand"] == "Il33") &
    (res["target"] == "ILC2")
].sort_values("lr_means", ascending=False)

In [ ]:
res[
    (res["ligand"] == "Il15") &
    (res["target"] == "ILC2")
].sort_values("lr_means", ascending=False)

In [ ]:
res[
    (res["ligand"] == "Il15") &
    (res["target"] == "NK_cell")
].sort_values("lr_means", ascending=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, fisher_exact
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()   # use your immune AnnData
gene = "Il1rl1"               # IL-33 receptor (ST2)
ilc2_label = "ILC2"           # change if your celltype label is different
pos_thresh = 0.5              # threshold for calling a cell Il1rl1+

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_ilc2 = adata[adata.obs["celltype"] == ilc2_label].copy()

x = adata_ilc2[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_ilc2 = pd.DataFrame({
    gene: x,
    "condition": adata_ilc2.obs["condition"].values
})

# -----------------------------
# 1A. TEST EXPRESSION LEVEL
# -----------------------------
ex_expr = df_ilc2.loc[df_ilc2["condition"] == "exercise", gene]
sed_expr = df_ilc2.loc[df_ilc2["condition"] == "sedentary", gene]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in ILC2")
print(f"Exercise mean:   {ex_expr.mean():.4f}")
print(f"Sedentary mean:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE
# -----------------------------
df_ilc2[f"{gene}_pos"] = df_ilc2[gene] > pos_thresh

ex_pos = df_ilc2.loc[df_ilc2["condition"] == "exercise", f"{gene}_pos"]
sed_pos = df_ilc2.loc[df_ilc2["condition"] == "sedentary", f"{gene}_pos"]

table = np.array([
    [ex_pos.sum(), (~ex_pos).sum()],
    [sed_pos.sum(), (~sed_pos).sum()]
])

oddsratio, prop_pval = fisher_exact(table)

print(f"\n{gene}+ proportion in ILC2 (threshold > {pos_thresh})")
print(f"Exercise proportion:   {ex_pos.mean():.4f}")
print(f"Sedentary proportion:  {sed_pos.mean():.4f}")
print(f"Fisher's exact p-value: {prop_pval:.4e}")
print("Contingency table:")
print(table)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il1rl1"
ilc2_label = "ILC2"
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_ilc2 = adata[adata.obs["celltype"] == ilc2_label].copy()

x = adata_ilc2[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_ilc2 = pd.DataFrame({
    gene: x,
    "condition": adata_ilc2.obs["condition"].values,
    "mouse_id": adata_ilc2.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_ilc2[f"{gene}_pos"] = df_ilc2[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_ilc2.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in ILC2 (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in ILC2 (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in ILC2")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ ILC2 proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il1rap"
ilc2_label = "ILC2"
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_ilc2 = adata[adata.obs["celltype"] == ilc2_label].copy()

x = adata_ilc2[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_ilc2 = pd.DataFrame({
    gene: x,
    "condition": adata_ilc2.obs["condition"].values,
    "mouse_id": adata_ilc2.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_ilc2[f"{gene}_pos"] = df_ilc2[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_ilc2.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in ILC2 (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in ILC2 (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in ILC2")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ ILC2 proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, fisher_exact

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()   # use your immune AnnData
gene = "Il15"               # IL15 
neutrophil_label = "Neutrophils"           # change if your celltype label is different
pos_thresh = 0.25              # threshold for calling a cell Il15+

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_neutrophil = adata[adata.obs["celltype"] == neutrophil_label].copy()

x = adata_neutrophil[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_neutrophil = pd.DataFrame({
    gene: x,
    "condition": adata_neutrophil.obs["condition"].values
})

# -----------------------------
# 1A. TEST EXPRESSION LEVEL
# -----------------------------
ex_expr = df_neutrophil.loc[df_neutrophil["condition"] == "exercise", gene]
sed_expr = df_neutrophil.loc[df_neutrophil["condition"] == "sedentary", gene]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in Neutrophils")
print(f"Exercise mean:   {ex_expr.mean():.4f}")
print(f"Sedentary mean:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE
# -----------------------------
df_neutrophil[f"{gene}_pos"] = df_neutrophil[gene] > pos_thresh

ex_pos = df_neutrophil.loc[df_neutrophil["condition"] == "exercise", f"{gene}_pos"]
sed_pos = df_neutrophil.loc[df_neutrophil["condition"] == "sedentary", f"{gene}_pos"]

table = np.array([
    [ex_pos.sum(), (~ex_pos).sum()],
    [sed_pos.sum(), (~sed_pos).sum()]
])

oddsratio, prop_pval = fisher_exact(table)

print(f"\n{gene}+ proportion in Neutrophils (threshold > {pos_thresh})")
print(f"Exercise proportion:   {ex_pos.mean():.4f}")
print(f"Sedentary proportion:  {sed_pos.mean():.4f}")
print(f"Fisher's exact p-value: {prop_pval:.4e}")
print("Contingency table:")
print(table)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [ex_expr.mean(), sed_expr.mean()]
prop_vals = [ex_pos.mean(), sed_pos.mean()]

pos_counts = [ex_pos.sum(), sed_pos.sum()]
total_counts = [len(ex_pos), len(sed_pos)]
color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}

colors = [color_map[c] for c in conditions]
# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression")
axes[0].set_title(f"Mean {gene} expression in Neutrophils")

ymax1 = max(mean_vals)
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.3)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells")
axes[1].set_title(f"{gene}+ Neutrophils proportion\n(threshold > {pos_thresh})")

for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.005, f"{pos}/{total}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals)
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.35)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, fisher_exact

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()   # use your immune AnnData
gene = "Il2rb"               # IL15 
neutrophil_label = "NK_cell"           # change if your celltype label is different
pos_thresh = 0.25              # threshold for calling a cell Il15+

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_neutrophil = adata[adata.obs["celltype"] == neutrophil_label].copy()

x = adata_neutrophil[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_neutrophil = pd.DataFrame({
    gene: x,
    "condition": adata_neutrophil.obs["condition"].values
})

# -----------------------------
# 1A. TEST EXPRESSION LEVEL
# -----------------------------
ex_expr = df_neutrophil.loc[df_neutrophil["condition"] == "exercise", gene]
sed_expr = df_neutrophil.loc[df_neutrophil["condition"] == "sedentary", gene]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in Neutrophils")
print(f"Exercise mean:   {ex_expr.mean():.4f}")
print(f"Sedentary mean:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE
# -----------------------------
df_neutrophil[f"{gene}_pos"] = df_neutrophil[gene] > pos_thresh

ex_pos = df_neutrophil.loc[df_neutrophil["condition"] == "exercise", f"{gene}_pos"]
sed_pos = df_neutrophil.loc[df_neutrophil["condition"] == "sedentary", f"{gene}_pos"]

table = np.array([
    [ex_pos.sum(), (~ex_pos).sum()],
    [sed_pos.sum(), (~sed_pos).sum()]
])

oddsratio, prop_pval = fisher_exact(table)

print(f"\n{gene}+ proportion in Neutrophils (threshold > {pos_thresh})")
print(f"Exercise proportion:   {ex_pos.mean():.4f}")
print(f"Sedentary proportion:  {sed_pos.mean():.4f}")
print(f"Fisher's exact p-value: {prop_pval:.4e}")
print("Contingency table:")
print(table)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [ex_expr.mean(), sed_expr.mean()]
prop_vals = [ex_pos.mean(), sed_pos.mean()]

pos_counts = [ex_pos.sum(), sed_pos.sum()]
total_counts = [len(ex_pos), len(sed_pos)]
color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}

colors = [color_map[c] for c in conditions]
# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression")
axes[0].set_title(f"Mean {gene} expression in NK cells")

ymax1 = max(mean_vals)
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.3)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells")
axes[1].set_title(f"{gene}+ NK cells proportion\n(threshold > {pos_thresh})")

for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.005, f"{pos}/{total}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals)
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.35)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il2rb"          # change as needed
nk_label = "NK_cell"    # change if your celltype label is different
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT NK DATA
# -----------------------------
adata_nk = adata[adata.obs["celltype"] == nk_label].copy()

x = adata_nk[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_nk = pd.DataFrame({
    gene: x,
    "condition": adata_nk.obs["condition"].values,
    "mouse_id": adata_nk.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_nk[f"{gene}_pos"] = df_nk[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_nk.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in NK cells (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in NK cells (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in NK cells")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ NK-cell proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il2rg"          # change as needed
nk_label = "NK_cell"    # change if your celltype label is different
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT NK DATA
# -----------------------------
adata_nk = adata[adata.obs["celltype"] == nk_label].copy()

x = adata_nk[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_nk = pd.DataFrame({
    gene: x,
    "condition": adata_nk.obs["condition"].values,
    "mouse_id": adata_nk.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_nk[f"{gene}_pos"] = df_nk[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_nk.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in NK cells (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in NK cells (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in NK cells")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ NK-cell proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il2rb"
ilc2_label = "ILC2"
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_ilc2 = adata[adata.obs["celltype"] == ilc2_label].copy()

x = adata_ilc2[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_ilc2 = pd.DataFrame({
    gene: x,
    "condition": adata_ilc2.obs["condition"].values,
    "mouse_id": adata_ilc2.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_ilc2[f"{gene}_pos"] = df_ilc2[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_ilc2.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in ILC2 (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in ILC2 (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in ILC2")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ ILC2 proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# SETTINGS
# -----------------------------
adata = adata_immune.copy()
gene = "Il2rg"
ilc2_label = "ILC2"
pos_thresh = 0.25

# -----------------------------
# CHECK GENE EXISTS
# -----------------------------
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# -----------------------------
# EXTRACT ILC2 DATA
# -----------------------------
adata_ilc2 = adata[adata.obs["celltype"] == ilc2_label].copy()

x = adata_ilc2[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df_ilc2 = pd.DataFrame({
    gene: x,
    "condition": adata_ilc2.obs["condition"].values,
    "mouse_id": adata_ilc2.obs["mouse_id"].values
})

# -----------------------------
# DEFINE POSITIVE CELLS
# -----------------------------
df_ilc2[f"{gene}_pos"] = df_ilc2[gene] > pos_thresh

# -----------------------------
# MOUSE-LEVEL SUMMARIES
# -----------------------------
mouse_summary = (
    df_ilc2.groupby(["mouse_id", "condition"])
    .agg(
        mean_expr=(gene, "mean"),
        prop_pos=(f"{gene}_pos", "mean"),
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

# -----------------------------
# 1A. TEST MEAN EXPRESSION (MOUSE LEVEL)
# -----------------------------
ex_expr = mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"]
sed_expr = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"]

expr_stat, expr_pval = mannwhitneyu(ex_expr, sed_expr, alternative="two-sided")

print(f"{gene} expression in ILC2 (mouse-level)")
print(f"Exercise mean of mouse means:   {ex_expr.mean():.4f}")
print(f"Sedentary mean of mouse means:  {sed_expr.mean():.4f}")
print(f"Mann-Whitney U p-value: {expr_pval:.4e}")

# -----------------------------
# 1B. TEST PROPORTION POSITIVE (MOUSE LEVEL)
# -----------------------------
ex_prop = mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"]
sed_prop = mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"]

prop_stat, prop_pval = mannwhitneyu(ex_prop, sed_prop, alternative="two-sided")

print(f"\n{gene}+ proportion in ILC2 (mouse-level; threshold > {pos_thresh})")
print(f"Exercise mean mouse proportion:   {ex_prop.mean():.4f}")
print(f"Sedentary mean mouse proportion:  {sed_prop.mean():.4f}")
print(f"Mann-Whitney U p-value: {prop_pval:.4e}")

print("\nMouse-level summary:")
print(mouse_summary)

# -----------------------------
# HELPER: p-value to stars
# -----------------------------
def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# -----------------------------
# PREP DATA FOR BAR PLOTS
# -----------------------------
conditions = ["exercise", "sedentary"]
condition_labels = ["Exercise", "Sedentary"]

mean_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "mean_expr"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "mean_expr"].mean()
]

prop_vals = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "prop_pos"].mean(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "prop_pos"].mean()
]

# pooled counts shown only as descriptive labels
pos_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "pos_count"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "pos_count"].sum()
]
total_counts = [
    mouse_summary.loc[mouse_summary["condition"] == "exercise", "total_cells"].sum(),
    mouse_summary.loc[mouse_summary["condition"] == "sedentary", "total_cells"].sum()
]

color_map = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}
colors = [color_map[c] for c in conditions]

# -----------------------------
# PLOT
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ---- Panel 1: mean expression ----
axes[0].bar(condition_labels, mean_vals, color=colors)
axes[0].set_ylabel(f"Mean {gene} expression per mouse")
axes[0].set_title(f"Mean {gene} expression in ILC2")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "mean_expr"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[0].scatter(x_jitter, vals, color="black", s=35, zorder=3)

ymax1 = max(mean_vals) if max(mean_vals) > 0 else 0.1
line_y1 = ymax1 * 1.08
text_y1 = ymax1 * 1.14

axes[0].plot([0, 0, 1, 1], [line_y1*0.98, line_y1, line_y1, line_y1*0.98], lw=1.5, c="black")
axes[0].text(
    0.5,
    text_y1,
    f"p = {expr_pval:.2e}\n{p_to_stars(expr_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[0].set_ylim(0, ymax1 * 1.35)

# ---- Panel 2: positive proportion ----
axes[1].bar(condition_labels, prop_vals, color=colors)
axes[1].set_ylabel(f"Proportion {gene}+ cells per mouse")
axes[1].set_title(f"{gene}+ ILC2 proportion\n(threshold > {pos_thresh})")

# overlay individual mouse points
for i, cond in enumerate(conditions):
    vals = mouse_summary.loc[mouse_summary["condition"] == cond, "prop_pos"].values
    x_jitter = np.random.normal(i, 0.04, size=len(vals))
    axes[1].scatter(x_jitter, vals, color="black", s=35, zorder=3)

# pooled counts just for reference
for i, (prop, pos, total) in enumerate(zip(prop_vals, pos_counts, total_counts)):
    axes[1].text(i, prop + 0.01, f"{int(pos)}/{int(total)}", ha="center", va="bottom", fontsize=10)

ymax2 = max(prop_vals) if max(prop_vals) > 0 else 0.1
line_y2 = ymax2 * 1.10
text_y2 = ymax2 * 1.18

axes[1].plot([0, 0, 1, 1], [line_y2*0.98, line_y2, line_y2, line_y2*0.98], lw=1.5, c="black")
axes[1].text(
    0.5,
    text_y2,
    f"p = {prop_pval:.2e}\n{p_to_stars(prop_pval)}",
    ha="center",
    va="bottom",
    fontsize=10
)
axes[1].set_ylim(0, ymax2 * 1.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# -----------------------------
# EXTRACT WHOLE IMMUNE DATA
# -----------------------------
x = adata[:, gene].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

df = pd.DataFrame({
    gene: x,
    "celltype": adata.obs["celltype"].values,
    "condition": adata.obs["condition"].values
})

df[f"{gene}_pos"] = df[gene] > pos_thresh

celltypes = df["celltype"].unique().tolist()
results = []

for ct in celltypes:
    sub = df[df["celltype"] == ct].copy()

    ex = sub[sub["condition"] == "exercise"][f"{gene}_pos"]
    sed = sub[sub["condition"] == "sedentary"][f"{gene}_pos"]

    # skip if one group is empty
    if len(ex) == 0 or len(sed) == 0:
        continue

    table = np.array([
        [ex.sum(), (~ex).sum()],
        [sed.sum(), (~sed).sum()]
    ])

    oddsratio, pval = fisher_exact(table)

    results.append({
        "celltype": ct,
        "exercise_prop": ex.mean(),
        "sedentary_prop": sed.mean(),
        "delta": ex.mean() - sed.mean(),
        "oddsratio": oddsratio,
        "pval": pval,
        "exercise_n": len(ex),
        "sedentary_n": len(sed),
        "exercise_pos_n": int(ex.sum()),
        "sedentary_pos_n": int(sed.sum())
    })

results_df = pd.DataFrame(results)

# multiple testing correction
results_df["padj"] = multipletests(results_df["pval"], method="fdr_bh")[1]

# sort by delta
results_df = results_df.sort_values("delta", ascending=False)

print(results_df)

In [ ]:
import scanpy as sc

sc.pp.normalize_total(adata_immune)
sc.pp.log1p(adata_immune)

In [ ]:
adata_immune = adata_immune.copy()

import liana as li

li.mt.cellphonedb(
    adata_immune,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    use_raw=False,
    n_perms=1000
)

In [ ]:
res = adata_immune.uns["liana_res"]
res.head()

In [ ]:
res[
    (res["ligand"] == "Il15") &
    (res["target"] == "ILC2")
].sort_values("lr_means", ascending=False)

In [ ]:
res[
    (res["ligand"] == "Il15") &
    (res["target"] == "ILC2")
].sort_values("lr_means", ascending=False)

In [ ]:
pivot = res.pivot_table(
    index="source",
    columns="target",
    values="lr_means",
    aggfunc="mean"
)

plt.figure(figsize=(6,5))
sns.heatmap(pivot, cmap="Reds")
plt.title("Cell–cell communication strength")
plt.show()

In [ ]:
res = adata_immune.uns["liana_res"]

il15 = res[res["ligand"]=="Il15"]

il15.sort_values("lr_means", ascending=False).head(10)

In [ ]:
neuts = adata_lr[adata_lr.obs["celltype"] == "Neutrophils"]

(neuts.to_df()["Il15"] > 0).groupby(neuts.obs["condition"]).mean()

In [ ]:
ilc2s = adata_lr[adata_lr.obs["celltype"] == "ILC2"]

(ilc2s.to_df()["Il15"] > 0).groupby(ilc2s.obs["condition"]).mean()

In [ ]:
ilc2s = adata_lr[adata_lr.obs["celltype"] == "ILC2"]

(ilc2s.to_df()["Il15ra"] > 0).groupby(ilc2s.obs["condition"]).mean()

In [ ]:
ilc2s = adata_lr[adata_lr.obs["celltype"] == "ILC2"]

(ilc2s.to_df()["Il2rg"] > 0).groupby(ilc2s.obs["condition"]).mean()

In [ ]:
adata_ex = adata_lr[adata_lr.obs["condition"]=="exercise"].copy()
adata_sed = adata_lr[adata_lr.obs["condition"]=="sedentary"].copy()

In [ ]:
li.mt.cellphonedb(
    adata_ex,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    use_raw=False,
    n_perms=1000
)
lr_ex = adata_ex.uns["liana_res"]

In [ ]:
li.mt.cellphonedb(
    adata_sed,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    use_raw=False,
    n_perms=1000
)
lr_sed = adata_sed.uns["liana_res"]

In [ ]:
lr_ex_il15 = lr_ex[
    (lr_ex["ligand"]=="Il15") &
    (lr_ex["source"]=="Neutrophils") &
    (lr_ex["target"]=="ILC2")
]

lr_sed_il15 = lr_sed[
    (lr_sed["ligand"]=="Il15") &
    (lr_sed["source"]=="Neutrophils") &
    (lr_sed["target"]=="ILC2")
]

In [ ]:
print(lr_ex_il15[["ligand","receptor","lr_means","cellphone_pvals"]])
print(lr_sed_il15[["ligand","receptor","lr_means","cellphone_pvals"]])

In [ ]:
lr_ex_il15["condition"] = "exercise"
lr_sed_il15["condition"] = "sedentary"

lr_compare = pd.concat([lr_ex_il15, lr_sed_il15])

In [ ]:
receptors = ["Il2ra","Il15ra","Il2rg","Il2rb"]

sc.pl.dotplot(
    adata_lr[adata_lr.obs["celltype"]=="ILC2"],
    receptors,
    groupby="condition"
)

In [ ]:
adata_lr.obs.groupby(["celltype","condition"]).size()

In [ ]:
sc.pl.dotplot(
    adata_immune[adata_immune.obs["celltype"]=="Neutrophils"],
    ["Il15"],
    groupby="condition"
)

In [ ]:
palette = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"}

In [ ]:

# color palette
palette = {
    "exercise": "#1f77b4",
    "sedentary": "#ff7f0e"
}

# rebuild counts table from scratch
counts = adata_immune.obs.groupby(["mouse_id", "celltype"]).size().unstack().fillna(0)

counts["total"] = counts.sum(axis=1)
counts["neut_prop"] = counts["Neutrophils"] / counts["total"]
counts["ilc2_prop"] = counts["ILC2"] / counts["total"]

# add condition back per mouse
mouse_condition = (
    adata_immune.obs[["mouse_id", "condition"]]
    .drop_duplicates()
)

counts = counts.reset_index().merge(mouse_condition, on="mouse_id", how="left")

# plot
plt.figure(figsize=(5,5))

sns.scatterplot(
    data=counts,
    x="neut_prop",
    y="ilc2_prop",
    hue="condition",
    palette=palette,
    s=100
)

plt.xlabel("Neutrophil proportion")
plt.ylabel("ILC2 proportion")
plt.title("Neutrophils vs ILC2 per mouse")
plt.legend(title="Condition")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------
# SETTINGS
# --------------------------
sample_col = "mouse_id"
condition_col = "condition"
fap_label = "FAP"      # change if needed
ilc2_label = "ILC2"

# --------------------------
# FAP proportion per mouse
# --------------------------
fap_counts = (
    adata_fap.obs
    .groupby([sample_col, condition_col, "celltype"])
    .size()
    .reset_index(name="n")
)

fap_totals = (
    adata_fap.obs
    .groupby([sample_col, condition_col])
    .size()
    .reset_index(name="total")
)

fap_props = fap_counts.merge(fap_totals, on=[sample_col, condition_col])
fap_props["prop"] = fap_props["n"] / fap_props["total"]

fap_only = fap_props[fap_props["celltype"] == fap_label][
    [sample_col, condition_col, "prop"]
].rename(columns={"prop": "fap_prop"})

# --------------------------
# ILC2 proportion per mouse
# --------------------------
immune_counts = (
    adata_immune.obs
    .groupby([sample_col, condition_col, "celltype"])
    .size()
    .reset_index(name="n")
)

immune_totals = (
    adata_immune.obs
    .groupby([sample_col, condition_col])
    .size()
    .reset_index(name="total")
)

immune_props = immune_counts.merge(immune_totals, on=[sample_col, condition_col])
immune_props["prop"] = immune_props["n"] / immune_props["total"]

ilc2_only = immune_props[immune_props["celltype"] == ilc2_label][
    [sample_col, condition_col, "prop"]
].rename(columns={"prop": "ilc2_prop"})

# --------------------------
# Merge by mouse
# --------------------------
plot_df = fap_only.merge(ilc2_only, on=[sample_col, condition_col], how="inner")

print(plot_df)

# --------------------------
# Plot
# --------------------------
colors = {
    "exercise": "#0072B2",
    "sedentary": "#E69F00"
}

plt.figure(figsize=(6, 6))

for cond in ["exercise", "sedentary"]:
    sub = plot_df[plot_df[condition_col] == cond]
    plt.scatter(
        sub["fap_prop"],
        sub["ilc2_prop"],
        s=80,
        color=colors[cond],
        label=cond
    )

    # Optional: label points by mouse
    for _, row in sub.iterrows():
        plt.text(
            row["fap_prop"] + 0.003,
            row["ilc2_prop"] + 0.001,
            row[sample_col],
            fontsize=9
        )

plt.xlabel("FAP proportion")
plt.ylabel("ILC2 proportion")
plt.title("FAPs vs ILC2 per mouse")
plt.legend(title="Condition")
plt.tight_layout()
plt.show()

# ILC2 and FAPS

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import liana as li

# -----------------------------
# 1. Keep only shared genes
# -----------------------------
shared_genes = adata_fap.var_names.intersection(adata_immune.var_names)

adata_fap_sub = adata_fap[:, shared_genes].copy()
adata_immune_sub = adata_immune[:, shared_genes].copy()

# -----------------------------
# 2. Make sure celltype + condition columns exist
# -----------------------------
for adata_obj in [adata_fap_sub, adata_immune_sub]:
    if "celltype" not in adata_obj.obs.columns:
        raise ValueError("Need a 'celltype' column in .obs")
    if "condition" not in adata_obj.obs.columns:
        raise ValueError("Need a 'condition' column in .obs")

# Optional: add source label
adata_fap_sub.obs["compartment"] = "FAP"
adata_immune_sub.obs["compartment"] = "Immune"

# -----------------------------
# 3. Concatenate into one object
# -----------------------------
adata_combined = ad.concat(
    [adata_fap_sub, adata_immune_sub],
    join="inner",
    label="dataset",
    keys=["fap", "immune"],
    merge="same"
)

# Make sure celltype is categorical
adata_combined.obs["celltype"] = adata_combined.obs["celltype"].astype("category")
adata_combined.obs["condition"] = adata_combined.obs["condition"].astype("category")

print(adata_combined)
print(adata_combined.obs["celltype"].value_counts())

In [ ]:
# Exercise only
adata_ex = adata_combined[adata_combined.obs["condition"] == "exercise"].copy()

li.mt.cellphonedb(
    adata_ex,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    verbose=True,
    use_raw=False,
    n_perms=1000
)


lr_ex = adata_ex.uns["liana_res"].copy()
print(lr_ex.head())

In [ ]:
# Sedentary only
adata_sed = adata_combined[adata_combined.obs["condition"] == "sedentary"].copy()

li.mt.cellphonedb(
    adata_sed,
    groupby="celltype",
    resource_name="mouseconsensus",
    expr_prop=0.1,
    verbose=True,
    use_raw=False,
     n_perms=1000
)

lr_sed = adata_sed.uns["liana_res"].copy()
print(lr_sed.head())

In [ ]:
# Exercise
il33_ex = lr_ex[
    (lr_ex["ligand"] == "Il33") &
    (lr_ex["receptor_complex"].fillna("").str.contains("Il1rl1")) &
    (lr_ex["receptor_complex"].fillna("").str.contains("Il1rap"))
].copy()

# Sedentary
il33_sed = lr_sed[
    (lr_sed["ligand"] == "Il33") &
    (lr_sed["receptor_complex"].fillna("").str.contains("Il1rl1")) &
    (lr_sed["receptor_complex"].fillna("").str.contains("Il1rap"))
].copy()

print("Exercise IL33 receptor-complex interactions:")
print(il33_ex.sort_values("lr_means", ascending=False).head(20))

print("\nSedentary IL33 receptor-complex interactions:")
print(il33_sed.sort_values("lr_means", ascending=False).head(20))

In [ ]:
def get_il33_signaling(df):
    rc = df["receptor_complex"].fillna("")
    return df[
        (df["ligand"] == "Il33") &
        rc.str.contains("Il1rl1") &
        rc.str.contains("Il1rap")
    ].copy()

# Apply
il33_ex = get_il33_signaling(lr_ex)
il33_sed = get_il33_signaling(lr_sed)

il33_ex["condition"] = "exercise"
il33_sed["condition"] = "sedentary"

# Combine
il33_all = pd.concat([il33_ex, il33_sed], axis=0)

# Keep FAP senders
fap_types = ["CD142+", "Prg4+", "Cxcl14+", "IPC_Skm", "Sca1-", "FAP"]
il33_all = il33_all[il33_all["source"].isin(fap_types)].copy()

print(il33_all[["condition", "source", "target", "lr_means", "cellphone_pvals"]]
      .sort_values(["condition", "lr_means"], ascending=[True, False]))

In [ ]:
for cond in ["exercise", "sedentary"]:
    sub = il33_all[il33_all["condition"] == cond].copy()
    
    heat = sub.pivot_table(
        index="source",
        columns="target",
        values="lr_means",
        aggfunc="mean"
    )
    
    heat = heat.reindex(fap_types)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(heat, annot=True, cmap="viridis")
    plt.title(f"{cond}: Il33 → Il1rl1 interaction strength (lr_means)")
    plt.xlabel("Target cell type")
    plt.ylabel("FAP sender subtype")
    plt.tight_layout()
    plt.show()

In [ ]:
compare_df = il33_all.pivot_table(
    index=["source", "target"],
    columns="condition",
    values="lr_means",
    aggfunc="mean"
).reset_index()

compare_df["delta_ex_minus_sed"] = compare_df["exercise"].fillna(0) - compare_df["sedentary"].fillna(0)

print(compare_df.sort_values("delta_ex_minus_sed", ascending=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Assume il33_all was already made using the CORRECT filter:
# ligand == "Il33" and receptor_complex contains both Il1rl1 and Il1rap
# -----------------------------

# Prepare data
compare_df = il33_all.pivot_table(
    index=["source", "target"],
    columns="condition",
    values="lr_means",
    aggfunc="mean"
).reset_index()

# Fill missing values
compare_df["exercise"] = compare_df["exercise"].fillna(0)
compare_df["sedentary"] = compare_df["sedentary"].fillna(0)

# Pair labels
compare_df["pair"] = compare_df["source"] + " → " + compare_df["target"]
compare_df["delta"] = compare_df["exercise"] - compare_df["sedentary"]

# Sort by exercise strength
compare_df = compare_df.sort_values(
    by="exercise",
    ascending=False
).reset_index(drop=True)

# Plot
x = np.arange(len(compare_df))
width = 0.35

plt.figure(figsize=(14, 6))
blue = "#1f77b4"
orange = "#ff7f0e"

plt.bar(x - width/2, compare_df["exercise"], width, color=blue, label="Exercise")
plt.bar(x + width/2, compare_df["sedentary"], width, color=orange, label="Sedentary")

plt.xticks(x, compare_df["pair"], rotation=45, ha="right")
plt.ylabel("Interaction strength (lr_means)")
plt.title("IL33 → IL1RL1/IL1RAP signaling by condition")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# 1. Filter LIANA results for Il15
# --------------------------
il15_ex = lr_ex[
    (lr_ex["ligand"] == "Il15") &
    (lr_ex["receptor"] == "Il15ra")
].copy()

il15_sed = lr_sed[
    (lr_sed["ligand"] == "Il15") &
    (lr_sed["receptor"] == "Il15ra")
].copy()

il15_ex["condition"] = "exercise"
il15_sed["condition"] = "sedentary"

# Combine
il15_all = pd.concat([il15_ex, il15_sed], ignore_index=True)

print("Exercise Il15 rows:", il15_ex.shape[0])
print("Sedentary Il15 rows:", il15_sed.shape[0])

# --------------------------
# 2. Keep strongest interaction per source-target-condition
#    in case multiple Il15 receptor-related entries exist
# --------------------------
il15_all = (
    il15_all.sort_values("lr_means", ascending=False)
    .drop_duplicates(subset=["source", "target", "condition"])
    .copy()
)

# --------------------------
# 3. Pivot lr_means into exercise/sedentary columns
# --------------------------
compare_df = (
    il15_all.pivot_table(
        index=["source", "target"],
        columns="condition",
        values="lr_means",
        aggfunc="first"
    )
    .reset_index()
)

# Make sure both columns exist
if "exercise" not in compare_df.columns:
    compare_df["exercise"] = 0
if "sedentary" not in compare_df.columns:
    compare_df["sedentary"] = 0

compare_df["exercise"] = compare_df["exercise"].fillna(0)
compare_df["sedentary"] = compare_df["sedentary"].fillna(0)

# --------------------------
# 4. Labels
# --------------------------
compare_df["pair"] = compare_df["source"] + " → " + compare_df["target"]
compare_df["delta"] = compare_df["exercise"] - compare_df["sedentary"]

# --------------------------
# 5. Order target groups
# --------------------------
target_order = {
    "ILC2": 0,
    "T_cell": 1,
    "T cells": 1,
    "Macrophages": 2,
    "Neutrophils": 3,
    "Mast_cell": 4
}

compare_df["target_rank"] = compare_df["target"].map(target_order).fillna(99)

compare_df = compare_df.sort_values(
    by=["target_rank", "exercise"],
    ascending=[True, False]
).reset_index(drop=True)

print(compare_df[["source", "target", "exercise", "sedentary", "delta"]])

# --------------------------
# 6. Plot
# --------------------------
x = np.arange(len(compare_df))
width = 0.35

plt.figure(figsize=(14, 6))
blue = "#1f77b4"
orange = "#ff7f0e"

plt.bar(x - width/2, compare_df["exercise"], width, color=blue, label="exercise")
plt.bar(x + width/2, compare_df["sedentary"], width, color=orange, label="sedentary")

# separator lines between target groups
target_changes = compare_df["target"].ne(compare_df["target"].shift()).to_numpy()
group_starts = np.where(target_changes)[0]

for idx in group_starts[1:]:
    plt.axvline(idx - 0.5, color="gray", linestyle="--", alpha=0.6)

plt.xticks(x, compare_df["pair"], rotation=45, ha="right")
plt.ylabel("Interaction strength (lr_means)")
plt.title("Il15 interactions by condition")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
celltype_col = "celltype"
condition_col = "condition"
ilc2_label = "ILC2"
genes = ["Il2rg", "Il2rb"]

# -----------------------------
# SUBSET TO ILC2
# -----------------------------
adata_ilc2 = adata_immune[adata_immune.obs[celltype_col] == ilc2_label].copy()

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
df = pd.DataFrame({
    "condition": adata_ilc2.obs[condition_col].values
})

for gene in genes:
    x = adata_ilc2[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray()
    df[gene] = np.ravel(x)

# binary expression
for gene in genes:
    df[f"{gene}_pos"] = df[gene] > 0

# -----------------------------
# CALCULATE PROPORTIONS
# -----------------------------
prop_df = (
    df.groupby("condition")[[f"{g}_pos" for g in genes]]
    .mean()
    .T
)

prop_df.columns = prop_df.columns.astype(str)
prop_df.index = genes

# -----------------------------
# STATISTICS
# -----------------------------
pvals = []

for gene in genes:
    ex = df.loc[df["condition"] == "exercise", f"{gene}_pos"]
    sed = df.loc[df["condition"] == "sedentary", f"{gene}_pos"]

    table = np.array([
        [ex.sum(), (~ex).sum()],
        [sed.sum(), (~sed).sum()]
    ])

    _, p = fisher_exact(table)
    pvals.append(p)

padj = multipletests(pvals, method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "gene": genes,
    "pval": pvals,
    "padj": padj
}).set_index("gene")

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_df.join(stats_df)
print(results)

# -----------------------------
# PLOT
# -----------------------------
x = np.arange(len(genes))
width = 0.35

blue = "#1f77b4"
orange = "#ff7f0e"

plt.figure(figsize=(7, 5))
plt.bar(x - width/2, results["exercise"], width, color=blue, label="exercise")
plt.bar(x + width/2, results["sedentary"], width, color=orange, label="sedentary")

# stars
ymax = np.maximum(results["exercise"], results["sedentary"])
for i, gene in enumerate(genes):
    y = ymax.iloc[i]
    offset = y * 0.02 if y > 0 else 0.01
    plt.text(i, y + offset, results.loc[gene, "sig"], ha="center", va="bottom", fontsize=12)

plt.xticks(x, genes)
plt.ylabel("Proportion of ILC2 cells expressing gene")
plt.xlabel("Gene")
plt.title("Proportion of ILC2 cells expressing Il2rg and Il2rb")
plt.legend(title="condition")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# 1. Define immune cell types
#    Edit these to match your labels exactly
# --------------------------
immune_cells = ["Macrophages", "Neutrophils", "T_cell", "B_cell", "ILC2", "Mast_cell", "NK_cell"]

# --------------------------
# 2. Filter IL15 interactions in each condition
#    Include both simple Il15ra and receptor complexes containing Il15ra
# --------------------------
il15_ex = lr_ex[
    (lr_ex["ligand"] == "Il15") &
    (
        (lr_ex["receptor"] == "Il15ra") |
        (lr_ex["receptor_complex"].fillna("").str.contains("Il15ra", case=False))
    ) &
    (lr_ex["source"].isin(immune_cells)) &
    (lr_ex["target"].isin(immune_cells))
].copy()

il15_sed = lr_sed[
    (lr_sed["ligand"] == "Il15") &
    (
        (lr_sed["receptor"] == "Il15ra") |
        (lr_sed["receptor_complex"].fillna("").str.contains("Il15ra", case=False))
    ) &
    (lr_sed["source"].isin(immune_cells)) &
    (lr_sed["target"].isin(immune_cells))
].copy()

il15_ex["condition"] = "exercise"
il15_sed["condition"] = "sedentary"

print("Exercise immune→immune IL15 rows:", il15_ex.shape[0])
print("Sedentary immune→immune IL15 rows:", il15_sed.shape[0])

# --------------------------
# 3. Combine
# --------------------------
il15_all = pd.concat([il15_ex, il15_sed], ignore_index=True)

# Keep strongest interaction per source-target-condition
il15_all = (
    il15_all.sort_values("lr_means", ascending=False)
    .drop_duplicates(subset=["source", "target", "condition"])
    .copy()
)

# --------------------------
# 4. Pivot for comparison
# --------------------------
compare_df = (
    il15_all.pivot_table(
        index=["source", "target"],
        columns="condition",
        values="lr_means",
        aggfunc="first"
    )
    .reset_index()
)

if "exercise" not in compare_df.columns:
    compare_df["exercise"] = 0
if "sedentary" not in compare_df.columns:
    compare_df["sedentary"] = 0

compare_df["exercise"] = compare_df["exercise"].fillna(0)
compare_df["sedentary"] = compare_df["sedentary"].fillna(0)

# Add p-values
pvals = (
    il15_all.groupby(["source", "target"])["cellphone_pvals"]
    .min()
    .reset_index()
)

compare_df = compare_df.merge(pvals, on=["source", "target"], how="left")

# Labels
compare_df["pair"] = compare_df["source"] + " → " + compare_df["target"]
compare_df["delta"] = compare_df["exercise"] - compare_df["sedentary"]

def p_to_star(p):
    if pd.isna(p):
        return "ns"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

compare_df["sig"] = compare_df["cellphone_pvals"].apply(p_to_star)

# Sort by strongest exercise interaction
compare_df = compare_df.sort_values(
    by=["exercise", "sedentary"],
    ascending=[False, False]
).reset_index(drop=True)

print(compare_df[["source", "target", "exercise", "sedentary", "cellphone_pvals"]])

# --------------------------
# 5. Plot
# --------------------------
x = np.arange(len(compare_df))
width = 0.35

plt.figure(figsize=(max(12, len(compare_df) * 0.6), 6))
blue = "#1f77b4"
orange = "#ff7f0e"

plt.bar(x - width/2, compare_df["exercise"], width, color=blue, label="exercise")
plt.bar(x + width/2, compare_df["sedentary"], width, color=orange, label="sedentary")

# significance stars
ymax = np.maximum(compare_df["exercise"], compare_df["sedentary"])
for i in range(len(compare_df)):
    y = ymax.iloc[i]
    offset = y * 0.03 if y > 0 else 0.01
    plt.text(i, y + offset, compare_df["sig"].iloc[i],
             ha="center", va="bottom", fontsize=11)

plt.xticks(x, compare_df["pair"], rotation=60, ha="right")
plt.ylabel("Interaction strength (lr_means)")
plt.title("Immune → immune Il15 interactions by condition")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Keep ALL interactions where target is ILC2
ilc2_ex = lr_ex[lr_ex["target"] == "ILC2"].copy()
ilc2_sed = lr_sed[lr_sed["target"] == "ILC2"].copy()

ilc2_ex["condition"] = "exercise"
ilc2_sed["condition"] = "sedentary"

ilc2_all = pd.concat([ilc2_ex, ilc2_sed], ignore_index=True)

print(ilc2_all.head())

In [ ]:
summary = (
    ilc2_all
    .groupby(["source", "condition"])["lr_means"]
    .sum()
    .unstack()
    .fillna(0)
    .reset_index()
)

summary["delta"] = summary["exercise"] - summary["sedentary"]

summary = summary.sort_values("exercise", ascending=False)

print(summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_df = summary.copy()

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(8,5))

plt.bar(x - width/2, plot_df["exercise"], width, label="exercise")
plt.bar(x + width/2, plot_df["sedentary"], width, label="sedentary")

plt.xticks(x, plot_df["source"], rotation=45, ha="right")
plt.ylabel("Total interaction strength (sum lr_means)")
plt.title("Total signaling into ILC2 by source")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Get top changing interactions into ILC2
pivot_lr = ilc2_all.pivot_table(
    index=["source", "ligand", "receptor"],
    columns="condition",
    values="lr_means",
    aggfunc="mean"
).reset_index()

pivot_lr["exercise"] = pivot_lr["exercise"].fillna(0)
pivot_lr["sedentary"] = pivot_lr["sedentary"].fillna(0)

pivot_lr["delta"] = pivot_lr["exercise"] - pivot_lr["sedentary"]

# Top increases
top_up = pivot_lr.sort_values("delta", ascending=False).head(20)

# Top decreases
top_down = pivot_lr.sort_values("delta").head(20)

print("Top increased signaling into ILC2:")
print(top_up)

print("\nTop decreased signaling into ILC2:")
print(top_down)

In [ ]:
import seaborn as sns

top_pairs = pd.concat([
    top_up.head(10),
    top_down.head(10)
])

top_pairs["pair"] = top_pairs["source"] + " | " + top_pairs["ligand"] + "→" + top_pairs["receptor"]

heatmap_df = top_pairs.set_index("pair")[["exercise", "sedentary"]]

plt.figure(figsize=(6,8))
sns.heatmap(heatmap_df, cmap="coolwarm", center=0)
plt.title("Top signaling pathways into ILC2")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il15": adata_immune[:, ["Il15"]].X.toarray().flatten(),
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

# Mean expression per mouse within each celltype
mouse_means = (
    df.groupby(["celltype", "condition", "mouse_id"])["Il15"]
    .mean()
    .reset_index()
)

# Condition-level mean of mouse-level means
mean_table = (
    mouse_means.groupby(["celltype", "condition"])["Il15"]
    .mean()
    .unstack()
)

# Keep a consistent order
celltypes = mean_table.index.tolist()

# Statistical testing at mouse level
pvals = []

for ct in celltypes:
    sub = mouse_means[mouse_means["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15"]
    sed = sub[sub["condition"] == "sedentary"]["Il15"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il15 expression per mouse")
plt.xlabel("Immune cell type")
plt.title("Mean Il15 expression by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add significance labels
ymax = mean_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il15ra": adata_immune[:, ["Il15ra"]].X.toarray().flatten(),
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

# Mean expression per mouse within each celltype
mouse_means = (
    df.groupby(["celltype", "condition", "mouse_id"])["Il15ra"]
    .mean()
    .reset_index()
)

# Condition-level mean of mouse-level means
mean_table = (
    mouse_means.groupby(["celltype", "condition"])["Il15ra"]
    .mean()
    .unstack()
)

# Keep a consistent order
celltypes = mean_table.index.tolist()

# Statistical testing at mouse level
pvals = []

for ct in celltypes:
    sub = mouse_means[mouse_means["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15ra"]
    sed = sub[sub["condition"] == "sedentary"]["Il15ra"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il15ra expression per mouse")
plt.xlabel("Immune cell type")
plt.title("Mean Il15ra expression by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add significance labels
ymax = mean_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.00002 if len(ymax) > 0 else 0.0002)

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il2rb": adata_immune[:, ["Il2rb"]].X.toarray().flatten(),
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

# Mean expression per mouse within each celltype
mouse_means = (
    df.groupby(["celltype", "condition", "mouse_id"])["Il2rb"]
    .mean()
    .reset_index()
)

# Condition-level mean of mouse-level means
mean_table = (
    mouse_means.groupby(["celltype", "condition"])["Il2rb"]
    .mean()
    .unstack()
)

celltypes = mean_table.index.tolist()

# Statistical testing at mouse level
pvals = []
for ct in celltypes:
    sub = mouse_means[mouse_means["celltype"] == ct]
    ex = sub[sub["condition"] == "exercise"]["Il2rb"]
    sed = sub[sub["condition"] == "sedentary"]["Il2rb"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan
    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il2rb expression per mouse")
plt.xlabel("Immune cell type")
plt.title("Mean Il2rb expression by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

ymax = mean_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il2rg": adata_immune[:, ["Il2rg"]].X.toarray().flatten(),
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

# Mean expression per mouse within each celltype
mouse_means = (
    df.groupby(["celltype", "condition", "mouse_id"])["Il2rg"]
    .mean()
    .reset_index()
)

# Condition-level mean of mouse-level means
mean_table = (
    mouse_means.groupby(["celltype", "condition"])["Il2rg"]
    .mean()
    .unstack()
)

celltypes = mean_table.index.tolist()

# Statistical testing at mouse level
pvals = []
for ct in celltypes:
    sub = mouse_means[mouse_means["celltype"] == ct]
    ex = sub[sub["condition"] == "exercise"]["Il2rg"]
    sed = sub[sub["condition"] == "sedentary"]["Il2rg"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan
    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il2rg expression per mouse")
plt.xlabel("Immune cell type")
plt.title("Mean Il2rg expression by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

ymax = mean_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.02 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
gene = "Il15"          # change to "Il15ra", "Il2rb", or "Il2rg"
threshold = 0.25       # or 0.5 if that is what you are using

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
expr = adata_immune[:, gene].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    gene: expr,
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

df[f"{gene}_pos"] = df[gene] > threshold

# -----------------------------
# PROPORTION POSITIVE PER MOUSE
# -----------------------------
mouse_props = (
    df.groupby(["celltype", "condition", "mouse_id"])[f"{gene}_pos"]
    .mean()
    .reset_index()
)

# Mean of mouse-level proportions for plotting
prop_table = (
    mouse_props.groupby(["celltype", "condition"])[f"{gene}_pos"]
    .mean()
    .unstack()
)

celltypes = prop_table.index.tolist()

# -----------------------------
# STATISTICAL TESTING
# -----------------------------
pvals = []

for ct in celltypes:
    sub = mouse_props[mouse_props["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"][f"{gene}_pos"]
    sed = sub[sub["condition"] == "sedentary"][f"{gene}_pos"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)
print(results)

# -----------------------------
# OPTIONAL: descriptive pooled counts
# -----------------------------
count_df = (
    df.groupby(["celltype", "condition"])
    .agg(
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

count_table = count_df.pivot(index="celltype", columns="condition", values=["pos_count", "total_cells"])

# -----------------------------
# PLOT
# -----------------------------
ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel(f"Proportion of {gene}+ cells per mouse (>{threshold})")
plt.xlabel("Immune cell type")
plt.title(f"Proportion of {gene}+ cells by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add pooled counts as labels (descriptive only)
for i, ct in enumerate(prop_table.index):
    ex_pos = count_table.loc[ct, ("pos_count", "exercise")] if ("pos_count", "exercise") in count_table.columns else np.nan
    ex_tot = count_table.loc[ct, ("total_cells", "exercise")] if ("total_cells", "exercise") in count_table.columns else np.nan
    sed_pos = count_table.loc[ct, ("pos_count", "sedentary")] if ("pos_count", "sedentary") in count_table.columns else np.nan
    sed_tot = count_table.loc[ct, ("total_cells", "sedentary")] if ("total_cells", "sedentary") in count_table.columns else np.nan

# Add significance labels
ymax = prop_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
gene = "Il15ra"          # change to "Il15ra", "Il2rb", or "Il2rg"
threshold = 0.25       # or 0.5 if that is what you are using

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
expr = adata_immune[:, gene].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    gene: expr,
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

df[f"{gene}_pos"] = df[gene] > threshold

# -----------------------------
# PROPORTION POSITIVE PER MOUSE
# -----------------------------
mouse_props = (
    df.groupby(["celltype", "condition", "mouse_id"])[f"{gene}_pos"]
    .mean()
    .reset_index()
)

# Mean of mouse-level proportions for plotting
prop_table = (
    mouse_props.groupby(["celltype", "condition"])[f"{gene}_pos"]
    .mean()
    .unstack()
)

celltypes = prop_table.index.tolist()

# -----------------------------
# STATISTICAL TESTING
# -----------------------------
pvals = []

for ct in celltypes:
    sub = mouse_props[mouse_props["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"][f"{gene}_pos"]
    sed = sub[sub["condition"] == "sedentary"][f"{gene}_pos"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)
print(results)

# -----------------------------
# OPTIONAL: descriptive pooled counts
# -----------------------------
count_df = (
    df.groupby(["celltype", "condition"])
    .agg(
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

count_table = count_df.pivot(index="celltype", columns="condition", values=["pos_count", "total_cells"])

# -----------------------------
# PLOT
# -----------------------------
ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel(f"Proportion of {gene}+ cells per mouse (>{threshold})")
plt.xlabel("Immune cell type")
plt.title(f"Proportion of {gene}+ cells by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add pooled counts as labels (descriptive only)
for i, ct in enumerate(prop_table.index):
    ex_pos = count_table.loc[ct, ("pos_count", "exercise")] if ("pos_count", "exercise") in count_table.columns else np.nan
    ex_tot = count_table.loc[ct, ("total_cells", "exercise")] if ("total_cells", "exercise") in count_table.columns else np.nan
    sed_pos = count_table.loc[ct, ("pos_count", "sedentary")] if ("pos_count", "sedentary") in count_table.columns else np.nan
    sed_tot = count_table.loc[ct, ("total_cells", "sedentary")] if ("total_cells", "sedentary") in count_table.columns else np.nan

# Add significance labels
ymax = prop_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
gene = "Il2rb"          # change to "Il15ra", "Il2rb", or "Il2rg"
threshold = 0.25       # or 0.5 if that is what you are using

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
expr = adata_immune[:, gene].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    gene: expr,
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

df[f"{gene}_pos"] = df[gene] > threshold

# -----------------------------
# PROPORTION POSITIVE PER MOUSE
# -----------------------------
mouse_props = (
    df.groupby(["celltype", "condition", "mouse_id"])[f"{gene}_pos"]
    .mean()
    .reset_index()
)

# Mean of mouse-level proportions for plotting
prop_table = (
    mouse_props.groupby(["celltype", "condition"])[f"{gene}_pos"]
    .mean()
    .unstack()
)

celltypes = prop_table.index.tolist()

# -----------------------------
# STATISTICAL TESTING
# -----------------------------
pvals = []

for ct in celltypes:
    sub = mouse_props[mouse_props["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"][f"{gene}_pos"]
    sed = sub[sub["condition"] == "sedentary"][f"{gene}_pos"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)
print(results)

# -----------------------------
# OPTIONAL: descriptive pooled counts
# -----------------------------
count_df = (
    df.groupby(["celltype", "condition"])
    .agg(
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

count_table = count_df.pivot(index="celltype", columns="condition", values=["pos_count", "total_cells"])

# -----------------------------
# PLOT
# -----------------------------
ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel(f"Proportion of {gene}+ cells per mouse (>{threshold})")
plt.xlabel("Immune cell type")
plt.title(f"Proportion of {gene}+ cells by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add pooled counts as labels (descriptive only)
for i, ct in enumerate(prop_table.index):
    ex_pos = count_table.loc[ct, ("pos_count", "exercise")] if ("pos_count", "exercise") in count_table.columns else np.nan
    ex_tot = count_table.loc[ct, ("total_cells", "exercise")] if ("total_cells", "exercise") in count_table.columns else np.nan
    sed_pos = count_table.loc[ct, ("pos_count", "sedentary")] if ("pos_count", "sedentary") in count_table.columns else np.nan
    sed_tot = count_table.loc[ct, ("total_cells", "sedentary")] if ("total_cells", "sedentary") in count_table.columns else np.nan

# Add significance labels
ymax = prop_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -----------------------------
# SETTINGS
# -----------------------------
gene = "Il2rg"          # change to "Il15ra", "Il2rb", or "Il2rg"
threshold = 0.25       # or 0.5 if that is what you are using

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
expr = adata_immune[:, gene].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    gene: expr,
    "celltype": adata_immune.obs["celltype"].values,
    "condition": adata_immune.obs["condition"].values,
    "mouse_id": adata_immune.obs["mouse_id"].values
})

df[f"{gene}_pos"] = df[gene] > threshold

# -----------------------------
# PROPORTION POSITIVE PER MOUSE
# -----------------------------
mouse_props = (
    df.groupby(["celltype", "condition", "mouse_id"])[f"{gene}_pos"]
    .mean()
    .reset_index()
)

# Mean of mouse-level proportions for plotting
prop_table = (
    mouse_props.groupby(["celltype", "condition"])[f"{gene}_pos"]
    .mean()
    .unstack()
)

celltypes = prop_table.index.tolist()

# -----------------------------
# STATISTICAL TESTING
# -----------------------------
pvals = []

for ct in celltypes:
    sub = mouse_props[mouse_props["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"][f"{gene}_pos"]
    sed = sub[sub["condition"] == "sedentary"][f"{gene}_pos"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)
print(results)

# -----------------------------
# OPTIONAL: descriptive pooled counts
# -----------------------------
count_df = (
    df.groupby(["celltype", "condition"])
    .agg(
        pos_count=(f"{gene}_pos", "sum"),
        total_cells=(f"{gene}_pos", "size")
    )
    .reset_index()
)

count_table = count_df.pivot(index="celltype", columns="condition", values=["pos_count", "total_cells"])

# -----------------------------
# PLOT
# -----------------------------
ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel(f"Proportion of {gene}+ cells per mouse (>{threshold})")
plt.xlabel("Immune cell type")
plt.title(f"Proportion of {gene}+ cells by immune cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add pooled counts as labels (descriptive only)
for i, ct in enumerate(prop_table.index):
    ex_pos = count_table.loc[ct, ("pos_count", "exercise")] if ("pos_count", "exercise") in count_table.columns else np.nan
    ex_tot = count_table.loc[ct, ("total_cells", "exercise")] if ("total_cells", "exercise") in count_table.columns else np.nan
    sed_pos = count_table.loc[ct, ("pos_count", "sedentary")] if ("pos_count", "sedentary") in count_table.columns else np.nan
    sed_tot = count_table.loc[ct, ("total_cells", "sedentary")] if ("total_cells", "sedentary") in count_table.columns else np.nan

# Add significance labels
ymax = prop_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.002 if len(ymax) > 0 else 0.02)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()